[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/homework.ipynb)

# Homework: Day 1 → Day 2
**Due at the start of Day 2** | Estimated time: 1.5 hours

This homework consolidates Day 1 material and bridges to Day 2. It maps onto the new Day-1 structure:

- **Q1 — Multiple Testing Concepts (LO1, LO3)**. Exercises **Lecture 1** (hypothesis testing: Bonferroni / FWER, BH / FDR, $q$-values) and includes an **AI explanation + critique** exercise.
- **Q2 — FDR in Practice (LO1, LO3)**. Connects both Day-1 lectures: empirical-Bayes variance shrinkage from **Lecture 2**, BH-adjusted $p$-values and Storey's $\hat\pi_0$ from **Lecture 1**. **All three sub-parts ask you to use AI for code generation** and to critique what comes back. *You will implement* `limma`'s moderated-$t$ idea from scratch in Python (no R, no rpy2) — this makes the empirical-Bayes mechanism concrete in roughly 15 lines of numpy/scipy.
- **Q3 — Evaluating Analysis Output (LO3)**. Exercises the critical-evaluation thread from both lectures, with explicit **you-vs-AI comparison** exercises in writing specifications, debugging, and code review.

::: {.callout-note}
## How this notebook uses AI

This homework integrates AI at several steps, not just at the end. Two kinds of AI exercises appear:

- **Single-shot critiques** — you prompt, paste the response, and evaluate it against named criteria.
- **Multi-turn conversations** — you prompt, critique the response, write a follow-up prompt that addresses your critique, paste the refined response, and write a final assessment. *At least one multi-turn exercise per notebook is required.* In this homework, Q3(d) is the multi-turn exercise.

Use any AI assistant (the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, ChatGPT, or another). Note which AI you used for each prompt — different models behave differently, and the differences are part of the lesson.
:::

> **This notebook uses the Python (`hd-stats`) kernel** — the same kernel as Labs 1–4. No R, no Bioconductor, no kernel switching: the entire workshop runs on one Python stack.

In [ ]:
# Verify you are running the Python (hd-stats) kernel
import sys
print(f"Python {sys.version.split()[0]}")
import os
print(f"Working directory: {os.getcwd()}")


---
## Q1: Multiple Testing Concepts (LO1)

A proteomics experiment measures the abundance of $p = 5{,}000$ proteins in $n = 30$ tissue samples (15 tumor, 15 normal). For each protein, a two-sample $t$-test is performed.

### (a)
Under the global null (no protein is differentially abundant), how many proteins would you expect to reject at $\alpha = 0.05$ without any correction?

**YOUR ANSWER:**


### (b)
The Bonferroni-corrected threshold is $\alpha / m$. Compute this threshold. A protein with a "real" effect has a $p$-value of $3 \times 10^{-4}$. Would it survive Bonferroni correction?

**YOUR ANSWER:**



In [ ]:
# You can verify (a) and (b) here:
m = 5000
alpha = 0.05
expected_fp_uncorrected = m * alpha
bonf_threshold = alpha / m
print(f"Expected false positives (uncorrected): {expected_fp_uncorrected:.0f}")
print(f"Bonferroni threshold: {bonf_threshold:.2e}")
print(f"Real effect p = 3e-4 survives Bonferroni? {3e-4 < bonf_threshold}")


### (c)
You apply the BH procedure at FDR $= 0.10$ and get 120 rejections. Give an interpretation of this result in plain language that a biologist would understand.

**YOUR ANSWER:**


### (d)
Your collaborator says "We found 120 significant proteins." A reviewer replies "But 12 of those are probably false positives!" Is the reviewer being unreasonable? Explain.

**YOUR ANSWER:**


### (e) AI explanation + critique

Ask an AI assistant ([UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, or ChatGPT) the following question and paste its response below:

> "Explain the difference between FWER (Bonferroni-style) and FDR (Benjamini–Hochberg-style) control to a wet-lab biologist who has never taken a statistics course. Use the analogy of finding stars in a noisy telescope image. Three short paragraphs at most."

**AI's response:**


**Your critique** — evaluate the AI's response against these criteria:

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Does the AI define both FWER and FDR correctly? | | |
| Does it explain the key difference (any vs. fraction)? | | |
| Does it pick the right context (FDR for genomic discovery)? | | |
| Does it avoid the common error of saying "FDR is the probability that a finding is false"? | | |
| Is the star/telescope analogy accurate? | | |

**One-sentence overall assessment:** Would you use this explanation when introducing the concept to a wet-lab collaborator? Why or why not?

---
## Q2: FDR in Practice (LO1, LO3)

Using the Golub dataset (introduced in **Lecture 1** and used in **Labs 1 and 2**), perform the following analysis. You should use AI to write the Python code, but evaluate its correctness and interpret the results yourself.

The two empirical-Bayes ideas you will exercise here:

- **From Lecture 2**: empirical-Bayes shrinkage of gene-level variance estimates toward a common prior (the idea behind `limma`'s `eBayes()`), producing moderated $t$-statistics that are more stable than raw $t$-tests when $n$ is small.
- **From Lecture 1**: BH adjustment converts $p$-values to FDR-controlled discoveries. Storey's $\hat\pi_0$ from the same lecture provides an adaptive variant.

In this homework you will implement Smyth's moderated-$t$ idea *from scratch in Python* — no R, no rpy2. The implementation is roughly 15 lines of numpy + scipy, and it makes the empirical-Bayes mechanism concrete in a way that calling a library does not.

### Setup — load the Golub data and the standard scientific-Python stack

In [ ]:
# Q2 uses the same standard scientific-Python stack as the labs:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy import special
from statsmodels.stats.multitest import multipletests

print(f"numpy {np.__version__}, pandas {pd.__version__}")
print(f"scipy {stats.__name__.split('.')[0]} OK; statsmodels.multipletests OK")


In [ ]:
# Load the Golub expression matrix and metadata directly from the workshop repo
expr_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/main/data/golub_expression.csv"
meta_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/main/data/golub_metadata.csv"

expr = pd.read_csv(expr_url, index_col=0)            # 3,051 genes (rows) x 72 samples (cols)
meta = pd.read_csv(meta_url)                          # 72 rows: sample_id, subtype

print(f"Expression matrix: {expr.shape[0]} genes x {expr.shape[1]} samples")
print(f"Metadata rows: {meta.shape[0]}")
print("Class distribution:")
print(meta["subtype"].value_counts())

# Sanity check: columns of expr should match meta["sample_id"]
assert list(expr.columns) == list(meta["sample_id"]), "Sample IDs do not match."

# Group masks (boolean, length 72)
all_mask = (meta["subtype"].values == "ALL")
aml_mask = (meta["subtype"].values == "AML")
gene_names = expr.index.values
n_all = int(all_mask.sum())
n_aml = int(aml_mask.sum())
print(f"ALL: {n_all} samples; AML: {n_aml} samples")


### (a) BH-adjusted p-values via hand-rolled moderated $t$-statistics

Direct an AI assistant to implement the empirical-Bayes variance shrinkage of `limma`'s `eBayes` *from scratch in Python* (i.e., not by calling an R library), then BH-adjust the resulting moderated $p$-values. Report the number of genes significant at FDR $< 0.01$, $< 0.05$, $< 0.10$.

The moderated $t$-statistic is

$$\tilde t_g = \frac{\bar y_{g,A} - \bar y_{g,B}}{\tilde s_g\, \sqrt{1/n_A + 1/n_B}},
\qquad
\tilde s_g^2 = \frac{d_0\, s_0^2 + d_g\, s_g^2}{d_0 + d_g}$$

where $s_g^2$ is the per-gene pooled sample variance, $d_g = n_A + n_B - 2$ is its degrees of freedom, and $(d_0, s_0^2)$ are prior hyperparameters fit to the population of gene-level variances (a simple choice for the homework: $d_0 = 4$, $s_0^2 = \exp(\text{mean of }\log s_g^2)$). The reference distribution under the null is a $t$ with $d_0 + d_g$ degrees of freedom.

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file — just runnable code I can paste into a Colab cell). Given pandas DataFrames `expr` (3,051 genes x 72 samples) and `meta` (sample_id, subtype with values 'ALL' / 'AML'), implement Smyth's moderated $t$-statistic from scratch: compute per-gene pooled sample variance $s_g^2$ with degrees of freedom $d_g$, fit an inverse-chi-squared prior using the heuristic $d_0 = 4$ and $s_0^2 = \exp(\text{mean log }s_g^2)$, compute moderated variances $\tilde s_g^2 = (d_0 s_0^2 + d_g s_g^2)/(d_0 + d_g)$ and moderated $t$-statistics, get two-sided p-values from a $t$-distribution with $d_0 + d_g$ degrees of freedom, then BH-adjust via `statsmodels.stats.multitest.multipletests(..., method='fdr_bh')`. Print the counts of genes with adjusted $p$-value below 0.01, 0.05, and 0.10. Use numpy, scipy.stats, and statsmodels only — no R, no rpy2."

**Include the prompt you used and note whether the AI spontaneously produced the moderated-$t$ implementation, or whether you had to direct it.**

**Your prompt to the AI:**


**The AI you used:** (e.g., the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), ChatGPT, Claude)


**Did the AI produce the moderated-$t$ implementation in one shot, or did you have to direct it?**

In [ ]:
# Paste / refine the AI-generated moderated-t code here.
# Reference scaffold (you should still verify and refine):
#   1. Split the expression matrix by group:
#         A = expr.loc[:, all_mask].values   # 3051 x n_A
#         B = expr.loc[:, aml_mask].values   # 3051 x n_B
#   2. Per-gene pooled sample variance and degrees of freedom:
#         d_g = n_A + n_B - 2
#         s_g_sq = ((A - A.mean(axis=1, keepdims=True))**2).sum(axis=1) + \
#                  ((B - B.mean(axis=1, keepdims=True))**2).sum(axis=1)
#         s_g_sq /= d_g
#   3. Inverse-chi-squared prior (heuristic): d_0 = 4, s_0_sq = exp(mean(log(s_g_sq)))
#   4. Moderated variance and t-statistic:
#         s_tilde_sq = (d_0 * s_0_sq + d_g * s_g_sq) / (d_0 + d_g)
#         mean_diff  = A.mean(axis=1) - B.mean(axis=1)
#         t_mod      = mean_diff / np.sqrt(s_tilde_sq * (1/n_A + 1/n_B))
#   5. Two-sided p-values from t-distribution with d_0 + d_g dof:
#         p_mod = 2 * (1 - stats.t.cdf(np.abs(t_mod), df=d_0 + d_g))
#   6. BH adjustment:
#         _, p_adj, _, _ = multipletests(p_mod, alpha=0.05, method='fdr_bh')
#   7. Report counts at FDR thresholds 0.01, 0.05, 0.10.

# YOUR CODE HERE


### (b) Estimate $\hat\pi_0$ using Storey's formula

Estimate $\pi_0$ (the proportion of truly null genes) directly from the raw moderated $p$-values using Storey's adaptive estimator,

$$\hat\pi_0(\lambda) = \frac{\#\{p_i > \lambda\}}{m\,(1 - \lambda)},
\qquad \lambda = 0.5.$$

Lab 2 implemented the same recipe (cell 7). The homework version uses the moderated $p$-values from Q2(a).

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file). Given a numpy array `p_mod` of raw $p$-values from the moderated-$t$ analysis on the Golub data, compute Storey's $\hat\pi_0(\lambda)$ at $\lambda = 0.5$ using the formula $\hat\pi_0 = \#\{p_i > \lambda\} / (m(1-\lambda))$. Print $\hat\pi_0$ with three decimal places and a one-line interpretation: 'roughly X% of the 3,051 genes are estimated to be truly null.' Do not invent or hallucinate numbers — only print what the formula returns."

In [ ]:
# YOUR CODE HERE
# Hint:
#   lam = 0.5
#   pi0_hat = float(np.sum(p_mod > lam) / (len(p_mod) * (1 - lam)))
#   pi0_hat = min(pi0_hat, 1.0)
#   print(f"pi0_hat = {pi0_hat:.3f}; roughly {pi0_hat * 100:.1f}% of the 3,051 genes are estimated to be truly null.")


**Interpretation of $\hat\pi_0$:**



### (c) P-value histogram with annotation

Make a $p$-value histogram of the moderated $p$-values from Q2(a). Annotate it with two reference lines (the uniform-null density at 1 and the estimated null-density height $\hat\pi_0$) and shade two regions of the $p$-value axis (the spike of true alternatives near 0 and the flat null region near 1).

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file). Given a numpy array `p_mod` of moderated $p$-values from the Golub data and the Storey estimate `pi0_hat`, produce a matplotlib histogram of the $p$-values using 50 bins with density scaling. Overlay a horizontal dashed reference line at the uniform-null density (height 1) labelled 'expected if all null'. Overlay a horizontal dotted line at height `pi0_hat` labelled 'estimated null density'. Use `ax.axvspan(0, 0.1, alpha=0.15, color='C1')` to shade the spike-of-alternatives region and `ax.axvspan(0.5, 1.0, alpha=0.1, color='C0')` to shade the uniform-null region. Title the figure, label both axes, and add a legend."

In [ ]:
# YOUR CODE HERE
# Sketch (50 bins, density-scaled, two reference lines, two shaded regions):
#
#   fig, ax = plt.subplots(figsize=(8, 4))
#   ax.hist(p_mod, bins=50, density=True, color='steelblue', edgecolor='white')
#   ax.axhline(1.0, color='red', linestyle='--', label='expected if all null')
#   ax.axhline(pi0_hat, color='orange', linestyle=':', label=f'estimated null density (pi0 = {pi0_hat:.2f})')
#   ax.axvspan(0, 0.1, alpha=0.15, color='C1', label='spike of true alternatives')
#   ax.axvspan(0.5, 1.0, alpha=0.10, color='C0', label='roughly uniform null region')
#   ax.set_xlabel('moderated p-value'); ax.set_ylabel('density')
#   ax.set_title('Annotated histogram of moderated p-values (Golub ALL vs AML)')
#   ax.legend()
#   plt.tight_layout(); plt.show()


**One-paragraph interpretation of (a)–(c):**

(Discuss: How many discoveries at each FDR level? What does $\hat\pi_0$ suggest about the proportion of nulls? Does the histogram shape match your expectation? What was AI-generated vs. your own?)




---
## Q3: Evaluating Analysis Output (LO3)

### (a) Two specifications for the same task

Write **two** specifications for the same analysis task: *"Perform differential expression analysis on a microarray dataset comparing treatment and control groups."*

- **Specification A** (vague, likely to produce incorrect/incomplete analysis):


- **Specification B** (precise, likely to produce correct analysis on the first attempt — name the method, the test, the multiple-testing correction, and the threshold):


**What makes Specification B better?**


### (b) Verifying a citation

You read (in a paper or AI output) the claim:
> *"Gene XYZ promotes cell proliferation in breast cancer through the PI3K/AKT pathway (Smith et al., 2019)."*

Describe the steps you would take to verify this before including it in your own work.

**YOUR ANSWER:**


### (c) An error you encountered today

Describe one specific error you encountered during today's labs — whether in AI-generated code, your own code, or a collaborator's analysis.

- What was the error?
- How did you catch it?
- What statistical knowledge was required to recognize it?

**Suggested AI prompt for follow-up** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file — just runnable code I can paste into a Colab cell). Here is a snippet of analysis code: \[paste your snippet\]. Identify the most likely statistical error and write a corrected version of the snippet. If the original is correct, return the same code unchanged with a one-line comment confirming this."

**YOUR ANSWER:**


### (d) A short review — you vs. AI vs. AI-critiques-you (multi-turn)

This is the multi-turn AI exercise required for this homework.

**Turn 1 — your review.** You receive a complete differential expression analysis from a collaborator. It uses raw two-sample $t$-tests with BH correction on a microarray dataset with 72 samples. Write a short review (3–4 sentences) describing what's adequate and what should be changed, and why.

*(Hint: connect this to Lecture 2's argument that empirical-Bayes variance shrinkage — the moderated $t$-statistic — is preferred over raw $t$-tests when $n$ is small. The [Decision Guide](https://pflahert.github.io/hd-stats-workshop/syllabus/decision-guide.html) summarizes the recommended methods.)*

**YOUR REVIEW (Turn 1):**


**Turn 2 — AI's review.** Now ask an AI for its review of the same scenario:

> "Reply in three to four sentences only (no .docx file). A collaborator submitted a differential expression analysis on a microarray dataset with $n = 72$ samples using raw two-sample $t$-tests with Benjamini–Hochberg FDR correction. Write a brief critical review explaining what is adequate about this approach and what should be changed, and why."

**AI's review (Turn 2, paste here):**


**Turn 3 — AI critiques your review.** Now turn the AI loose on *your* review. Use the following prompt:

> "Reply in three to four sentences only (no .docx file). I wrote the following review of a colleague's differential expression analysis: \[paste your Turn-1 review verbatim\]. Critique my review for completeness, accuracy, and tone. Did I miss any important methodological issues that a statistical reviewer should flag? Did I overstate any concern?"

**AI's critique of your review (paste here):**


**Final synthesis (your work).** Combine the best points from all three turns into a single 3–5-sentence review you would actually send to the collaborator. State explicitly which points came from you, which from the AI's review of the scenario, and which came from the AI's critique of your review.

**FINAL SYNTHESIZED REVIEW:**


**Reflection.** What did this three-turn exchange teach you about using AI as a reviewer? Were the AI's contributions complementary to your own, or did they substantially overlap? Did the AI catch issues you missed, or did you catch issues the AI missed?

---
## Submission

Bring your completed notebook (or written answers + code outputs for Q2) to Day 2. Be prepared to reference your answers during Lecture 3 and Lecture 4 discussions.

### Assessment criteria

This is a formative assessment. Completeness and engagement matter more than getting every detail right. We are particularly interested in:

- Your interpretation of statistical results **in your own words** (the AI cells record AI output; the critique and assessment cells record *your* evaluation).
- Your ability to identify errors in analysis code or output (Q3(c), and the AI-critique cells throughout Q1 and Q2).
- The Q3(d) three-turn exchange: a thoughtful comparison of your review, the AI's review, and the AI's critique of you.

> **Tip:** `File → Download → Download .ipynb` to keep a local copy of your work.